# MEGNet Multi-Fidelidade para Materiais 2D do C2DB

Pipeline de descoberta de materiais 2D com UWBG (Eg > 3,4 eV).

## Dataset

- **C2DB** (Gjerding et al. 2021) — 16905 materiais 2D, formato ASE SQLite
- **Colunas de bandgap:** `gap` (PBE) e `gap_hse` (HSE06)
- **Amostras multi-fidelidade:** 8699 (PBE) + 3363 (HSE) = **12062 total**
- **Candidatos UWBG reais no C2DB:** 193 (PBE) e 467 (HSE) com Eg > 3,4 eV

## Estrutura

1. Carregamento do C2DB via ASE
2. Preparação do dataset multi-fidelidade: `gap` = PBE (fidelidade 0), `gap_hse` = HSE (fidelidade 1)
3. Treinamento do MEGNet do zero (baseline)
4. Fine-tuning a partir do modelo MP (transfer learning)
5. Avaliação por fidelidade
6. Triagem UWBG — predição HSE para todos os materiais

**Referências:**
- Chen et al. 2019 — MEGNet
- Chen et al. 2021 — Multi-fidelidade
- Gjerding et al. 2021 — C2DB
- Ko et al. 2025 — MatGL

In [ ]:
from __future__ import annotations

import glob
import json
import warnings
from copy import deepcopy
from functools import partial
from pathlib import Path

import ase.db
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from dgl.data.utils import split_dataset
from pymatgen.core import Structure
from pymatgen.io.ase import AseAtomsAdaptor

from matgl.config import DEFAULT_ELEMENTS
from matgl.ext.pymatgen import Structure2Graph
from matgl.graph.data import MGLDataLoader, MGLDataset, collate_fn_graph
from matgl.models import MEGNet
from matgl.utils.training import ModelLightningModule

warnings.simplefilter("ignore")


print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
_candidate_roots = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
FINAL_ROOT = next((p if p.name == 'final' else p / 'final' for p in _candidate_roots if p.name == 'final' or (p / 'final').is_dir()), None)
if FINAL_ROOT is None:
    raise RuntimeError(f'Não foi possível localizar final/ a partir de {NOTEBOOK_DIR}')
FINAL_ROOT = FINAL_ROOT.resolve()
if str(FINAL_ROOT) not in sys.path:
    sys.path.insert(0, str(FINAL_ROOT))

from common import DATA_DIR, FINAL_ROOT, REPO_ROOT, RUNS_DIR, ensure_run_dir, required_path

print(f'Final root: {FINAL_ROOT}')
if torch.cuda.is_available():
    torch.set_float32_matmul_precision('high')


In [ ]:
# ── Identificação do Run ─────────────────────────────────────────
RUN_ID   = "001"
RUN_NAME = "megnet_finetune_c2db"
RUN_DIR = ensure_run_dir(RUN_ID, RUN_NAME)
(RUN_DIR / "model" / "scratch").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "model" / "finetune").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "cache").mkdir(parents=True, exist_ok=True)

# ── Reprodutibilidade ────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Dados ────────────────────────────────────────────────────────
USE_JARVIS       = False  # mantém o pipeline final autocontido; ative apenas com cache local JARVIS
C2DB_PATH = required_path(DATA_DIR / "raw" / "c2db.db", "C2DB")
EHULL_TRAIN      = 0.2
EHULL_SCREEN     = 0.2
REQUIRE_DYN_STAB = False

# ── Grafo ────────────────────────────────────────────────────────
CUTOFF = 5.0

# ── Treino ───────────────────────────────────────────────────────
BATCH_SIZE           = 128
LR_SCRATCH           = 1e-3
LR_FINETUNE          = 1e-4
MAX_EPOCHS           = 360
PATIENCE_SCRATCH     = 35
PATIENCE_FINETUNE    = 35
MIN_DELTA_SCRATCH    = 1e-3
MIN_DELTA_FINETUNE   = 1e-3
WEIGHT_DECAY         = 1e-4
FORCE_RETRAIN        = True
ACCELERATOR          = "gpu" if torch.cuda.is_available() else "cpu"
DEVICES              = 1

# ── Triagem ──────────────────────────────────────────────────────
UWBG_THRESHOLD = 3.4

# ── Wandb ────────────────────────────────────────────────────────
USE_WANDB     = False
WANDB_PROJECT = "uwbg-discovery"

HPARAMS = dict(
    seed=SEED, ehull_train=EHULL_TRAIN, ehull_screen=EHULL_SCREEN,
    cutoff=CUTOFF, batch_size=BATCH_SIZE,
    lr_scratch=LR_SCRATCH, lr_finetune=LR_FINETUNE,
    max_epochs=MAX_EPOCHS,
    patience_scratch=PATIENCE_SCRATCH, patience_finetune=PATIENCE_FINETUNE,
    min_delta_scratch=MIN_DELTA_SCRATCH, min_delta_finetune=MIN_DELTA_FINETUNE,
    weight_decay=WEIGHT_DECAY, uwbg_threshold=UWBG_THRESHOLD,
)

print(f"Run      : {RUN_DIR.resolve()}")
print(f"C2DB     : {C2DB_PATH}")
print(f"Treino   : force_retrain={FORCE_RETRAIN} accelerator={ACCELERATOR}")

In [ ]:
from tqdm.auto import tqdm
from lightning.pytorch.callbacks import Callback
from lightning.pytorch.loggers import CSVLogger

if USE_WANDB:
    import wandb
    from lightning.pytorch.loggers import WandbLogger


class EpochProgress(Callback):
    """Barra de progresso por época: atualiza in-place com train/val/best."""

    def __init__(self, max_epochs: int, label: str = "treino"):
        self.max_epochs = max_epochs
        self.label = label
        self._pbar = None
        self._best = float("inf")

    def on_train_start(self, trainer, pl_module):
        self._best = float("inf")
        self._pbar = tqdm(
            total=self.max_epochs, desc=self.label,
            unit="ép", dynamic_ncols=True,
        )

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        m = trainer.callback_metrics
        val   = float(m.get("val_MAE",   float("nan")))
        train = float(m.get("train_MAE", float("nan")))
        if val < self._best:
            self._best = val
        self._pbar.update(1)
        self._pbar.set_postfix(
            train=f"{train:.4f}",
            val=f"{val:.4f}",
            best=f"{self._best:.4f}",
        )

    def on_train_end(self, trainer, pl_module):
        if self._pbar:
            self._pbar.close()
        print(f"✓ {self.label} — best val_MAE = {self._best:.4f} eV")


# Run wandb compartilhado entre scratch e finetune (evita finish/reinit na mesma sessão)
_wandb_run = None


def make_logger(model_name: str):
    """Retorna [WandbLogger, CSVLogger] ou CSVLogger dependendo de USE_WANDB."""
    global _wandb_run
    csv = CSVLogger(str(RUN_DIR / "logs"), name=f"c2db_{model_name}")
    if USE_WANDB:
        if _wandb_run is None:
            _wandb_run = wandb.init(
                project=WANDB_PROJECT,
                name=f"{RUN_ID}_{RUN_NAME}",
                config=HPARAMS,
            )
        wb = WandbLogger(experiment=_wandb_run, prefix=model_name)
        return [wb, csv]
    return csv

## 1. Carregamento do C2DB

O banco já está disponível localmente em formato ASE SQLite.

**Colunas relevantes do C2DB:**

| Coluna       | Descrição                        | Unidade |
|--------------|----------------------------------|---------|
| `gap`        | Bandgap PBE                      | eV      |
| `gap_hse`    | Bandgap HSE06                    | eV      |
| `gap_dir`    | Bandgap direto PBE               | eV      |
| `gap_dir_hse`| Bandgap direto HSE               | eV      |
| `ehull`      | Energia acima da convex hull     | eV/atom |
| `hform`      | Energia de formação              | eV/atom |
| `dyn_stab`   | Estável dinamicamente (`"Yes"`)  | —       |
| `uid`        | Identificador único              | —       |

**Nota:** A coluna PBE é `gap` (não `gap_pbe`).

In [ ]:
assert C2DB_PATH.exists(), f"C2DB não encontrado: {C2DB_PATH.resolve()}"

adaptor = AseAtomsAdaptor()

def load_c2db(
    db_path: Path,
    max_ehull: float = 0.2,
    require_dyn_stab: bool = False,
) -> tuple[dict, dict, dict]:
    """
    Lê o C2DB e retorna dicionários indexados pelo uid.

    - `gap`     → bandgap PBE  (pode ser None ou 0.0 para metais)
    - `gap_hse` → bandgap HSE  (subconjunto — todos com HSE também têm PBE)

    Filtros aplicados:
    - ehull <= max_ehull (estabilidade termodinâmica)
    - dyn_stab == 'Yes' (opcional)
    - pelo menos um valor de bandgap disponível
    """
    db = ase.db.connect(str(db_path))

    structures = {}
    gap_pbe = {}   # coluna 'gap' no C2DB
    gap_hse = {}   # coluna 'gap_hse' no C2DB
    skipped = 0

    for row in db.select():
        uid = row.get('uid') or str(row.id)

        # Filtro de estabilidade termodinâmica
        ehull = row.get('ehull')
        if ehull is not None and ehull > max_ehull:
            skipped += 1
            continue

        # Filtro de estabilidade dinâmica (opcional)
        if require_dyn_stab and row.get('dyn_stab') != 'Yes':
            skipped += 1
            continue

        g_pbe = row.get('gap')      # PBE bandgap
        g_hse = row.get('gap_hse')  # HSE06 bandgap

        if g_pbe is None and g_hse is None:
            skipped += 1
            continue

        try:
            atoms = row.toatoms()
            struct = adaptor.get_structure(atoms)
        except Exception:
            skipped += 1
            continue

        structures[uid] = struct
        gap_pbe[uid] = g_pbe
        gap_hse[uid] = g_hse

    print(f"Materiais carregados : {len(structures)}")
    print(f"  com PBE (gap)      : {sum(v is not None for v in gap_pbe.values())}")
    print(f"  com HSE (gap_hse)  : {sum(v is not None for v in gap_hse.values())}")
    print(f"  pulados            : {skipped}")
    return structures, gap_pbe, gap_hse


structures, gap_pbe, gap_hse = load_c2db(C2DB_PATH, max_ehull=EHULL_TRAIN, require_dyn_stab=REQUIRE_DYN_STAB)

### 1b. Carregar JARVIS-DFT 2D (dados extras de treino)

O JARVIS-DFT fornece ~800 materiais 2D com cálculos de bandgap em dois níveis:
- `optb88vdw_bandgap` → fidelidade 0 (OptB88vdW, nível PBE-like)
- `mbj_bandgap`       → fidelidade 1 (TB-mBJ, nível HSE-like)

**Importante:** materiais JARVIS são adicionados **apenas ao treino** —
o conjunto de teste permanece C2DB-only para manter comparabilidade com experimentos anteriores.

In [ ]:
if USE_JARVIS:
    from ase import Atoms as AseAtoms
    from jarvis.db.figshare import data as jarvis_data
    from jarvis.core.atoms import Atoms as JAtoms

    def _parse_float(val, min_val=0.01):
        """Converte campo JARVIS para float; retorna None se inválido ou abaixo do mínimo."""
        if val is None:
            return None
        try:
            v = float(val)
            return v if v >= min_val else None
        except (ValueError, TypeError):
            return None


    def load_jarvis_megnet(min_gap=0.01):
        """
        Carrega JARVIS-DFT 2D e retorna dicionários no mesmo formato do load_c2db:
          structures : uid -> pymatgen Structure
          gap_pbe    : uid -> float (OptB88vdW)
          gap_hse    : uid -> float (TB-mBJ) ou None

        UIDs recebem prefixo 'jarvis_' para evitar colisão com C2DB.
        """
        print('Baixando JARVIS-DFT 2D...')
        raw = jarvis_data('dft_2d')
        print(f'  Total bruto: {len(raw)} entradas')

        j_structures = {}
        j_gap_pbe    = {}
        j_gap_hse    = {}
        skipped = 0

        for entry in raw:
            g_pbe = _parse_float(entry.get('optb88vdw_bandgap'), min_gap)
            if g_pbe is None:  # exclui metais e entradas sem gap
                skipped += 1
                continue

            g_hse = _parse_float(entry.get('mbj_bandgap'), min_gap)
            jid = entry.get('jid', f'j{len(j_structures)}')
            uid = f'jarvis_{jid}'

            try:
                jatoms = JAtoms.from_dict(entry['atoms'])
                ase_atoms = AseAtoms(
                    symbols=jatoms.elements,
                    positions=jatoms.cart_coords,
                    cell=jatoms.lattice_mat,
                    pbc=[True, True, False],
                )
                struct = adaptor.get_structure(ase_atoms)
            except Exception:
                skipped += 1
                continue

            j_structures[uid] = struct
            j_gap_pbe[uid]    = g_pbe
            j_gap_hse[uid]    = g_hse

        n_hse = sum(v is not None for v in j_gap_hse.values())
        print(f'  Carregados : {len(j_structures)} materiais ({skipped} pulados)')
        print(f'  com OptB88 (PBE-like, fid=0): {len(j_structures)}')
        print(f'  com TB-mBJ (HSE-like, fid=1): {n_hse}')
        return j_structures, j_gap_pbe, j_gap_hse


    j_structures, j_gap_pbe, j_gap_hse = load_jarvis_megnet()

else:
    print('JARVIS desativado: usando apenas dados locais C2DB para manter final/ autocontido.')
    j_structures, j_gap_pbe, j_gap_hse = {}, {}, {}


In [ ]:
pbe_vals = [v for v in gap_pbe.values() if v is not None]
hse_vals = [v for v in gap_hse.values() if v is not None]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, vals, label, color in [
    (axes[0], pbe_vals, "PBE",  "steelblue"),
    (axes[1], hse_vals, "HSE",  "darkorange"),
]:
    ax.hist(vals, bins=70, color=color, edgecolor="white", linewidth=0.3)
    ax.axvline(3.4, color="red", linestyle="--", linewidth=1.5, label="UWBG (3,4 eV)")
    uwbg = sum(1 for v in vals if v > 3.4)
    ax.set_title(f"Bandgap {label} — {len(vals)} materiais ({uwbg} UWBG)")
    ax.set_xlabel(f"Bandgap {label} (eV)")
    ax.set_ylabel("Contagem")
    ax.legend()

plt.tight_layout()
plt.savefig(RUN_DIR / "figures/c2db_bandgap_distribution.png", dpi=150)
plt.show()

## 2. Dataset multi-fidelidade

Codificação das fidelidades como **state attribute** (atributo global do grafo):

| Fidelidade | ID | Funcional | N amostras |
|------------|----|-----------|-----------|
| PBE        | 0  | GGA-PBE   | ~8699     |
| HSE        | 1  | HSE06     | ~3363     |

O mesmo material pode aparecer duas vezes (uma por fidelidade disponível).

In [ ]:
FIDELITY_NAMES = {0: 'PBE', 1: 'HSE'}

all_structures = []
all_fidelities = []
all_targets    = []
all_uids       = []
all_sources    = []  # 'c2db' ou 'jarvis' — controla o split

# --- C2DB ---
for uid, struct in structures.items():
    g_pbe = gap_pbe.get(uid)
    if g_pbe is not None and g_pbe > 0:
        all_structures.append(deepcopy(struct))
        all_fidelities.append(0)
        all_targets.append(float(g_pbe))
        all_uids.append(f'{uid}_pbe')
        all_sources.append('c2db')

    g_hse = gap_hse.get(uid)
    if g_hse is not None:
        all_structures.append(deepcopy(struct))
        all_fidelities.append(1)
        all_targets.append(float(g_hse))
        all_uids.append(f'{uid}_hse')
        all_sources.append('c2db')

# --- JARVIS (apenas treino — marcado como 'jarvis') ---
for uid, struct in j_structures.items():
    g_pbe = j_gap_pbe.get(uid)
    if g_pbe is not None:
        all_structures.append(deepcopy(struct))
        all_fidelities.append(0)
        all_targets.append(float(g_pbe))
        all_uids.append(f'{uid}_pbe')
        all_sources.append('jarvis')

    g_hse = j_gap_hse.get(uid)
    if g_hse is not None:
        all_structures.append(deepcopy(struct))
        all_fidelities.append(1)
        all_targets.append(float(g_hse))
        all_uids.append(f'{uid}_hse')
        all_sources.append('jarvis')

n_c2db   = all_sources.count('c2db')
n_jarvis = all_sources.count('jarvis')
print(f'Total de amostras: {len(all_structures)}')
print(f'  C2DB  : {n_c2db}')
print(f'  JARVIS: {n_jarvis}')
for fid_id, fid_name in FIDELITY_NAMES.items():
    n = all_fidelities.count(fid_id)
    print(f'  {fid_name} (fidelidade {fid_id}): {n}')


## 3. MGLDataset

`Structure2Graph` com `cutoff=5.0 Å`. Para materiais 2D com vácuo (~15–20 Å no eixo c), o cutoff garante que não há ligações através do vácuo — o caráter 2D é preservado naturalmente.

In [ ]:
CACHE_DIR = str(RUN_DIR / "cache" / "C2DB_MGLDataset")

element_types = DEFAULT_ELEMENTS
cry_graph = Structure2Graph(element_types=element_types, cutoff=CUTOFF)

dataset = MGLDataset(
    structures=all_structures,
    graph_labels=all_fidelities,
    labels={"bandgap": all_targets},
    converter=cry_graph,
    filename=CACHE_DIR,
)
print(f"Dataset: {len(dataset)} amostras")

## 4. Split train/val/test e DataLoaders

In [ ]:
import numpy as np
from torch.utils.data import Subset

# Separa índices C2DB e JARVIS
c2db_idx   = [i for i, s in enumerate(all_sources) if s == 'c2db']
jarvis_idx = [i for i, s in enumerate(all_sources) if s == 'jarvis']

# Split 80/10/10 apenas nos índices C2DB
rng = np.random.default_rng(SEED)
c2db_idx_shuffled = rng.permutation(c2db_idx).tolist()
n = len(c2db_idx_shuffled)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

c2db_train_idx = c2db_idx_shuffled[:n_train]
c2db_val_idx   = c2db_idx_shuffled[n_train:n_train + n_val]
c2db_test_idx  = c2db_idx_shuffled[n_train + n_val:]

# Treino = C2DB train + todos os JARVIS
train_idx = c2db_train_idx + jarvis_idx
val_idx   = c2db_val_idx
test_idx  = c2db_test_idx

train_data = Subset(dataset, train_idx)
val_data   = Subset(dataset, val_idx)
test_data  = Subset(dataset, test_idx)

print(f'Train: {len(train_data)} (C2DB={len(c2db_train_idx)}, JARVIS={len(jarvis_idx)})')
print(f'Val  : {len(val_data)}  (C2DB-only)')
print(f'Test : {len(test_data)}  (C2DB-only)')

my_collate_fn = partial(collate_fn_graph, include_line_graph=False)

train_loader, val_loader, test_loader = MGLDataLoader(
    train_data=train_data,
    val_data=val_data,
    test_data=test_data,
    collate_fn=my_collate_fn,
    batch_size=BATCH_SIZE,
    num_workers=4,
    pin_memory=True,
)


In [ ]:
# ── Estatísticas do conjunto de treino ───────────────────────────
train_targets_list = [float(all_targets[i]) for i in train_idx]
train_fid_list     = [int(all_fidelities[i]) for i in train_idx]

n_total  = len(train_idx)
n_hse    = sum(1 for f in train_fid_list if f == 1)
n_danger = sum(1 for t in train_targets_list if 2.0 <= t <= 4.0)

print(f"Amostras de treino : {n_total}")
print(f"  HSE              : {n_hse} ({n_hse/n_total*100:.1f}%)")
print(f"  Gap 2–4 eV (PBE) : {n_danger} ({n_danger/n_total*100:.1f}%)")
print("Sampler            : uniforme (sem ponderação)")

## 5a. MEGNet — treinamento do zero (baseline)

Modelo treinado exclusivamente no C2DB.
- `ntypes_state=2` — apenas PBE e HSE
- `dim_state_embedding=64` — igual ao modelo MP, garante **100% de compatibilidade** para o fine-tuning
- EarlyStopping com `patience=30` para controlar overfitting

In [ ]:
def build_megnet_c2db(ntypes_state: int = 2, dim_state: int = 64) -> MEGNet:
    """
    Arquitetura MEGNet para o C2DB.
    dim_state=64 é idêntico ao modelo MP — garante 100% de compatibilidade
    de pesos no fine-tuning (apenas state_embedding é reinicializado).
    """
    return MEGNet(
        element_types=element_types,
        cutoff=CUTOFF,
        is_intensive=True,
        dim_state_embedding=dim_state,
        ntypes_state=ntypes_state,
        readout_type="set2set",
        include_states=True,
    )


model_scratch = build_megnet_c2db()
optimizer_scratch = torch.optim.Adam(model_scratch.parameters(), lr=LR_SCRATCH, weight_decay=WEIGHT_DECAY)
lit_scratch = ModelLightningModule(model=model_scratch, optimizer=optimizer_scratch, lr=LR_SCRATCH)
n_params = sum(p.numel() for p in model_scratch.parameters() if p.requires_grad)
print(f"Parâmetros treináveis: {n_params:,}")

In [ ]:
scratch_dir = RUN_DIR / "model" / "scratch"
existing_scratch = sorted(scratch_dir.glob("best-*.ckpt"))

if existing_scratch and not FORCE_RETRAIN:
    ckpt_path = existing_scratch[-1]
    print(f"Checkpoint scratch existente; pulando treino: {ckpt_path.name}")
    lit_scratch = ModelLightningModule.load_from_checkpoint(str(ckpt_path), model=model_scratch, map_location="cpu")
    model_scratch = lit_scratch.model
    trainer_scratch = L.Trainer(accelerator=ACCELERATOR, devices=DEVICES, logger=False, enable_progress_bar=False)
else:
    logger_scratch = make_logger("scratch")
    trainer_scratch = L.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator=ACCELERATOR,
        devices=DEVICES,
        enable_progress_bar=False,
        logger=logger_scratch,
        log_every_n_steps=20,
        callbacks=[
            EarlyStopping(monitor="val_MAE", patience=PATIENCE_SCRATCH, min_delta=MIN_DELTA_SCRATCH, mode="min", verbose=False),
            EpochProgress(MAX_EPOCHS, label="MEGNet scratch"),
            ModelCheckpoint(
                dirpath=str(scratch_dir),
                filename="best-{epoch:03d}-{val_MAE:.4f}",
                monitor="val_MAE",
                mode="min",
                save_top_k=1,
            ),
        ],
    )
    trainer_scratch.fit(model=lit_scratch, train_dataloaders=train_loader, val_dataloaders=val_loader)

print("" + "=" * 50)
print("MEGNet — treinado do zero (test set)")
trainer_scratch.test(model=lit_scratch, dataloaders=test_loader)

## 5b. MEGNet — fine-tuning a partir do modelo MP

O MEGNet treinado no Materials Project (~61k amostras, 4 fidelidades) aprendeu representações gerais
de estruturas cristalinas. O fine-tuning especializa esses pesos para o domínio 2D do C2DB.

**Estratégia de transferência:**
- `dim_state_embedding=64` em ambos os modelos → **106/106 camadas compatíveis** (100%)
- Apenas o `state_embedding` é reinicializado (MP usa 4×64, C2DB usa 2×64 — dimensões diferentes)
- Fine-tuning com `lr=1e-4` (10× menor) para preservar os pesos transferidos
- Checkpoint MP usado: `best-epoch=109` (val MAE=0.316 eV, treinado com EarlyStopping)


In [ ]:
MP_CKPT = None
pretrain_run = RUNS_DIR / "000_megnet_pretrain_mp"
best_ckpts = sorted((pretrain_run / "model" / "best").glob("*.ckpt"))
version_ckpts = sorted(pretrain_run.glob("version_*/checkpoints/*.ckpt"))

print("best/ encontrados   :", [str(p) for p in best_ckpts])
print("version/ encontrados:", [str(p) for p in version_ckpts])

if best_ckpts:
    MP_CKPT = str(best_ckpts[-1])
    print()
    print(f"✓ Usando melhor checkpoint MP: {MP_CKPT}")
elif version_ckpts:
    MP_CKPT = str(version_ckpts[-1])
    print()
    print(f"[!] best/ não encontrado. Usando último checkpoint: {MP_CKPT}")
else:
    print()
    print("[!] Nenhum checkpoint MP encontrado. Fine-tuning usará inicialização do zero.")


In [ ]:
def transfer_mp_weights(ckpt_path: str) -> MEGNet:
    """
    Carrega o modelo MP (ntypes_state=4, dim_state=64) e
    copia os pesos compatíveis para um novo modelo C2DB (ntypes_state=2, dim_state=32).
    """
    # Reconstrói a arquitetura do modelo MP
    model_mp = MEGNet(
        element_types=element_types,
        cutoff=CUTOFF,
        is_intensive=True,
        dim_state_embedding=64,
        ntypes_state=4,
        readout_type="set2set",
        include_states=True,
    )
    lit_mp = ModelLightningModule.load_from_checkpoint(
        ckpt_path, model=model_mp, map_location="cpu"
    )
    mp_weights = lit_mp.model.state_dict()

    # Modelo C2DB com dimensão menor
    model_c2db = build_megnet_c2db()
    c2db_weights = model_c2db.state_dict()

    transferred, skipped = 0, []
    for key in c2db_weights:
        if "state_embedding" in key:
            # Incompatível em forma — reinicialização aleatória
            skipped.append(key)
            continue
        if key in mp_weights and mp_weights[key].shape == c2db_weights[key].shape:
            c2db_weights[key] = mp_weights[key]
            transferred += 1
        else:
            skipped.append(key)

    model_c2db.load_state_dict(c2db_weights)
    print(f"Pesos transferidos  : {transferred}")
    print(f"Reinicializados     : {len(skipped)} — {skipped}")
    return model_c2db


if MP_CKPT is not None:
    model_ft = transfer_mp_weights(MP_CKPT)
else:
    print("Usando modelo do zero como fallback para o fine-tuning.")
    model_ft = build_megnet_c2db()

In [ ]:
optimizer_ft = torch.optim.Adam(model_ft.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
lit_ft = ModelLightningModule(model=model_ft, optimizer=optimizer_ft, lr=LR_FINETUNE)

finetune_dir = RUN_DIR / "model" / "finetune"
existing_ft = sorted(finetune_dir.glob("best-*.ckpt"))

if existing_ft and not FORCE_RETRAIN:
    ckpt_path = existing_ft[-1]
    print(f"Checkpoint fine-tune existente; pulando treino: {ckpt_path.name}")
    lit_ft = ModelLightningModule.load_from_checkpoint(str(ckpt_path), model=model_ft, map_location="cpu")
    model_ft = lit_ft.model
    trainer_ft = L.Trainer(accelerator=ACCELERATOR, devices=DEVICES, logger=False, enable_progress_bar=False)
else:
    logger_ft = make_logger("finetune")
    trainer_ft = L.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator=ACCELERATOR,
        devices=DEVICES,
        enable_progress_bar=False,
        logger=logger_ft,
        log_every_n_steps=20,
        callbacks=[
            EarlyStopping(monitor="val_MAE", patience=PATIENCE_FINETUNE, min_delta=MIN_DELTA_FINETUNE, mode="min", verbose=False),
            EpochProgress(MAX_EPOCHS, label="MEGNet finetune"),
            ModelCheckpoint(
                dirpath=str(finetune_dir),
                filename="best-{epoch:03d}-{val_MAE:.4f}",
                monitor="val_MAE",
                mode="min",
                save_top_k=1,
            ),
        ],
    )
    trainer_ft.fit(model=lit_ft, train_dataloaders=train_loader, val_dataloaders=val_loader)

print("" + "=" * 50)
print("MEGNet — fine-tuning (MP → C2DB) (test set)")
trainer_ft.test(model=lit_ft, dataloaders=test_loader)

## 6. Convergência — scratch vs fine-tuning

In [ ]:
def load_metrics(log_name: str) -> pd.DataFrame | None:
    files = sorted(glob.glob(str(RUN_DIR / f"logs/{log_name}/version_*/metrics.csv")))
    return pd.read_csv(files[-1]) if files else None


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs = [
    ("c2db_scratch",  "Do zero",     "steelblue",  "tomato"),
    ("c2db_finetune", "Fine-tuning", "darkorange", "mediumseagreen"),
]

for ax, (log_name, title, c_train, c_val) in zip(axes, configs):
    m = load_metrics(log_name)
    if m is None:
        ax.set_title(f"{title} — sem dados")
        continue
    m["train_MAE"].dropna().reset_index(drop=True).plot(ax=ax, color=c_train, label="Train MAE")
    m["val_MAE"].dropna().reset_index(drop=True).plot(ax=ax, color=c_val,   label="Val MAE")
    ax.axhline(0.3, color="gray", linestyle=":", alpha=0.7, label="0.3 eV")
    ax.set_xlabel("Época")
    ax.set_ylabel("MAE (eV)")
    ax.set_title(f"MEGNet C2DB — {title}")
    ax.legend()

plt.tight_layout()
plt.savefig(RUN_DIR / "figures/c2db_convergence.png", dpi=150)
plt.show()

## 7. Avaliação no conjunto de teste

In [ ]:
print("Scratch trainer :", getattr(trainer_scratch.state, "status", "loaded"))
print("Finetune trainer:", getattr(trainer_ft.state, "status", "loaded"))

In [ ]:
def evaluate_per_fidelity(
    lit_module: ModelLightningModule,
    test_data,
    all_uids: list[str],
    structures: dict,
) -> dict:
    """Calcula MAE e RMSE separado por fidelidade no conjunto de teste."""
    lit_module.model.eval()
    results = {fid: {"true": [], "pred": []} for fid in FIDELITY_NAMES}

    with torch.no_grad():
        for idx in range(len(test_data)):
            _, _, state, labels = test_data[idx]
            fid_id   = int(state)
            true_val = labels["bandgap"].item()

            orig_idx = test_data.indices[idx]
            base_uid = all_uids[orig_idx].rsplit("_", 1)[0]
            struct   = structures[base_uid]

            try:
                state_t  = torch.tensor([fid_id], dtype=torch.long)
                pred_val = lit_module.model.predict_structure(struct, state_attr=state_t).item()
            except Exception:
                continue

            results[fid_id]["true"].append(true_val)
            results[fid_id]["pred"].append(pred_val)

    print(f"{'Fidelidade':10s} {'N':>6s} {'MAE':>10s} {'RMSE':>10s}")
    print("-" * 40)
    for fid_id, fid_name in FIDELITY_NAMES.items():
        t = np.array(results[fid_id]["true"])
        p = np.array(results[fid_id]["pred"])
        if len(t) == 0:
            continue
        mae  = np.mean(np.abs(t - p))
        rmse = np.sqrt(np.mean((t - p) ** 2))
        print(f"{fid_name:10s} {len(t):>6d} {mae:>10.4f} {rmse:>10.4f}")
    return results


print("\nScratch — por fidelidade:")
res_scratch = evaluate_per_fidelity(lit_scratch, test_data, all_uids, structures)

print("\nFine-tuning — por fidelidade:")
res_ft = evaluate_per_fidelity(lit_ft, test_data, all_uids, structures)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
colors = {0: "steelblue", 1: "darkorange"}

for row_idx, (res, title) in enumerate([(res_scratch, "Do zero"), (res_ft, "Fine-tuning")]):
    for col_idx, (fid_id, fid_name) in enumerate(FIDELITY_NAMES.items()):
        ax = axes[row_idx][col_idx]
        t = np.array(res[fid_id]["true"])
        p = np.array(res[fid_id]["pred"])
        if len(t) == 0:
            continue
        ax.scatter(t, p, alpha=0.35, s=8, color=colors[fid_id])
        lim = [min(t.min(), p.min()) - 0.5, max(t.max(), p.max()) + 0.5]
        ax.plot(lim, lim, "k--", lw=1, alpha=0.6, label="Ideal")
        ax.axvline(3.4, color="red", ls=":", lw=1, alpha=0.6)
        ax.axhline(3.4, color="red", ls=":", lw=1, alpha=0.6, label="UWBG 3,4 eV")
        mae = np.mean(np.abs(t - p))
        ax.set_title(f"{title} — {fid_name} (MAE={mae:.3f} eV)")
        ax.set_xlabel(f"{fid_name} real (eV)")
        ax.set_ylabel(f"{fid_name} predito (eV)")
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(RUN_DIR / "figures/c2db_scatter.png", dpi=150)
plt.show()

## 8. Triagem UWBG

Prediz o bandgap HSE (fidelidade 1) para **todos** os materiais 2D do C2DB.
Filtra candidatos com Eg > 3,4 eV.

- O modelo usado é o **melhor entre scratch e fine-tuning** (menor val MAE)
- Metais confirmados (PBE gap = 0) são excluídos antes da predição

In [ ]:
HSE_FID = 1  # fidelidade HSE no state attribute

# Seleciona o melhor modelo entre scratch e fine-tuning
def best_val_mae(log_name: str) -> float:
    files = sorted(glob.glob(str(RUN_DIR / f"logs/{log_name}/version_*/metrics.csv")))
    if not files:
        return float("inf")
    m = pd.read_csv(files[-1])
    return m["val_MAE"].dropna().min()

val_scratch = best_val_mae("c2db_scratch")
val_ft      = best_val_mae("c2db_finetune")

if val_scratch <= val_ft:
    model_screen = lit_scratch.model
    model_label  = f"scratch (val MAE={val_scratch:.4f})"
else:
    model_screen = lit_ft.model
    model_label  = f"fine-tuning (val MAE={val_ft:.4f})"

print(f"Modelo selecionado para triagem: {model_label}")
model_screen.eval()

rows = []
with torch.no_grad():
    for uid, struct in structures.items():
        g_pbe = gap_pbe.get(uid)
        if g_pbe is not None and g_pbe == 0.0:
            continue

        try:
            state_t  = torch.tensor([HSE_FID], dtype=torch.long)
            pred_hse = model_screen.predict_structure(struct, state_attr=state_t).item()
        except Exception:
            continue

        rows.append({
            "uid":          uid,
            "formula":      struct.composition.reduced_formula,
            "n_atoms":      len(struct),
            "gap_pbe":      g_pbe,
            "gap_hse_true": gap_hse.get(uid),
            "gap_hse_pred": pred_hse,
            "uwbg":         pred_hse > UWBG_THRESHOLD,
        })

df = pd.DataFrame(rows).sort_values("gap_hse_pred", ascending=False).reset_index(drop=True)
df.to_csv(RUN_DIR / "outputs/uwbg_candidates.csv", index=False)

n_total = len(df)
n_uwbg  = df["uwbg"].sum()
print(f"Materiais triados   : {n_total}")
print(f"Candidatos UWBG     : {n_uwbg} ({n_uwbg / n_total * 100:.1f}%)")
print(f"Resultados salvos   : uwbg_candidates.csv")

In [ ]:
top = df[df["uwbg"]].head(20)
print(f"{'#':>3} {'uid':25} {'Fórmula':12} {'HSE pred':>10} {'HSE real':>10} {'PBE':>8}")
print("-" * 75)
for i, row in enumerate(top.itertuples(), 1):
    hse_r = f"{row.gap_hse_true:.3f}" if row.gap_hse_true is not None else "—"
    pbe_r = f"{row.gap_pbe:.3f}"       if row.gap_pbe      is not None else "—"
    print(f"{i:>3} {row.uid:25} {row.formula:12} {row.gap_hse_pred:>10.3f} {hse_r:>10} {pbe_r:>8}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df["gap_hse_pred"], bins=80, color="steelblue", edgecolor="white", lw=0.3, label="Todos")
ax.hist(
    df.loc[df["uwbg"], "gap_hse_pred"],
    bins=40, color="red", alpha=0.65, label=f"UWBG (>{UWBG_THRESHOLD} eV)"
)
ax.axvline(UWBG_THRESHOLD, color="red", linestyle="--", lw=1.5)
ax.set_xlabel("Bandgap HSE predito (eV)")
ax.set_ylabel("Contagem")
ax.set_title("Distribuição dos bandgaps HSE preditos — C2DB (fine-tuned)")
ax.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / "figures/uwbg_distribution.png", dpi=150)
plt.show()

## 8b. Avaliação de Classificação UWBG

Usa os **3027 materiais com HSE real disponível** como gabarito para avaliar o modelo como classificador binário (UWBG vs não-UWBG), variando o limiar de decisão.

In [ ]:
from sklearn.metrics import (
    confusion_matrix, precision_recall_curve,
    average_precision_score, ConfusionMatrixDisplay
)

# Subconjunto com HSE real disponível
df_known = df[df["gap_hse_true"].notna()].copy()
df_known["uwbg_true"] = df_known["gap_hse_true"] > UWBG_THRESHOLD
df_known["uwbg_pred"] = df_known["gap_hse_pred"] > UWBG_THRESHOLD

y_true  = df_known["uwbg_true"].astype(int).values
y_score = df_known["gap_hse_pred"].values   # score contínuo para a curva P-R

# --- Métricas no limiar 3.4 eV ---
tp = int(( df_known["uwbg_pred"] &  df_known["uwbg_true"]).sum())
fp = int(( df_known["uwbg_pred"] & ~df_known["uwbg_true"]).sum())
fn = int((~df_known["uwbg_pred"] &  df_known["uwbg_true"]).sum())
tn = int((~df_known["uwbg_pred"] & ~df_known["uwbg_true"]).sum())

precision_at = tp / (tp + fp) if (tp + fp) > 0 else 0
recall_at    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_at        = 2 * precision_at * recall_at / (precision_at + recall_at) if (precision_at + recall_at) > 0 else 0
ap           = average_precision_score(y_true, y_score)

print(f"Avaliação no limiar {UWBG_THRESHOLD} eV  ({len(df_known)} materiais com HSE real)")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  Precision : {precision_at:.3f}")
print(f"  Recall    : {recall_at:.3f}")
print(f"  F1        : {f1_at:.3f}")
print(f"  AP (area) : {ap:.3f}")
print()

# --- Figura: matriz de confusão + curva P-R ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Matriz de confusão
cm = confusion_matrix(df_known["uwbg_true"], df_known["uwbg_pred"])
disp = ConfusionMatrixDisplay(cm, display_labels=["Não-UWBG", "UWBG"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Matriz de Confusão — limiar {UWBG_THRESHOLD} eV")

# Curva Precision-Recall
precision_curve, recall_curve, thresholds_pr = precision_recall_curve(y_true, y_score)
axes[1].plot(recall_curve, precision_curve, color="darkorange", lw=2,
             label=f"AP = {ap:.3f}")
axes[1].scatter([recall_at], [precision_at], color="red", zorder=5, s=80,
                label=f"Limiar 3.4 eV (F1={f1_at:.3f})")
axes[1].axhline(tp / (tp + fp + fn + tn), color="gray", linestyle=":", alpha=0.5,
                label="Baseline (fração UWBG)")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Curva Precision-Recall — Classificação UWBG")
axes[1].legend()
axes[1].set_xlim(0, 1.02)
axes[1].set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(RUN_DIR / "figures/uwbg_precision_recall.png", dpi=150)
plt.show()

# --- Precision e Recall em diferentes limiares ---
print(f"{'Limiar (eV)':>12} {'Precision':>10} {'Recall':>10} {'F1':>8} {'UWBG pred':>10}")
print("-" * 55)
for thr in [3.5, 4.0, 3.4, 5.0, 5.5, 6.0]:
    pred = df_known["gap_hse_pred"] > thr
    true = df_known["uwbg_true"]
    tp_ = int(( pred &  true).sum())
    fp_ = int(( pred & ~true).sum())
    fn_ = int((~pred &  true).sum())
    p_ = tp_ / (tp_ + fp_) if (tp_ + fp_) > 0 else 0
    r_ = tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0
    f_ = 2*p_*r_/(p_+r_) if (p_+r_) > 0 else 0
    print(f"{thr:>12.1f} {p_:>10.3f} {r_:>10.3f} {f_:>8.3f} {tp_+fp_:>10}")

## 8c. Candidatos genuinamente novos

Materiais com `gap_hse_pred > 3.4 eV` que **não têm cálculo HSE real** no C2DB — são as predições genuinamente novas do modelo, candidatos para validação DFT.

In [ ]:
import ase.db

# Candidatos novos: UWBG predito e sem HSE real
df_new = df[df["uwbg"] & df["gap_hse_true"].isna()].copy()

# Enriquece com dados de estabilidade do C2DB
db = ase.db.connect(str(C2DB_PATH))
uid_to_stability = {}
for row in db.select():
    uid = row.get("uid") or str(row.id)
    uid_to_stability[uid] = {
        "ehull":    row.get("ehull"),
        "dyn_stab": row.get("dyn_stab"),
        "hform":    row.get("hform"),
    }

df_new["ehull"]    = df_new["uid"].map(lambda u: uid_to_stability.get(u, {}).get("ehull"))
df_new["dyn_stab"] = df_new["uid"].map(lambda u: uid_to_stability.get(u, {}).get("dyn_stab"))
df_new["hform"]    = df_new["uid"].map(lambda u: uid_to_stability.get(u, {}).get("hform"))

# Aplica filtro de estabilidade termodinâmica (EHULL_SCREEN)
df_new = df_new[df_new["ehull"].notna() & (df_new["ehull"] <= EHULL_SCREEN)].copy()
print(f"Candidatos novos (UWBG sem HSE, ehull≤{EHULL_SCREEN}): {len(df_new)}")

# Ordena por predição HSE decrescente
df_new = df_new.sort_values("gap_hse_pred", ascending=False).reset_index(drop=True)

print(f"Total de candidatos novos (sem HSE real): {len(df_new)}")
print(f"  Dinamicamente estáveis (dyn_stab=Yes) : {(df_new['dyn_stab'] == 'Yes').sum()}")
print(f"  ehull < 0.1 eV/atom                   : {(df_new['ehull'] < 0.1).sum()}")
print()

# Top 20 candidatos novos
print(f"{'#':>3} {'uid':25} {'Fórmula':12} {'HSE pred':>10} {'PBE':>8} {'ehull':>8} {'dyn_stab':>10}")
print("-" * 85)
for i, row in df_new.head(20).iterrows():
    pbe  = f"{row.gap_pbe:.3f}"   if row.gap_pbe  is not None else "—"
    eh   = f"{row.ehull:.4f}"     if pd.notna(row.ehull)      else "—"
    dyn  = row.dyn_stab           if pd.notna(row.dyn_stab)   else "—"
    print(f"{i+1:>3} {row.uid:25} {row.formula:12} {row.gap_hse_pred:>10.3f} {pbe:>8} {eh:>8} {dyn:>10}")

In [ ]:
# Priorização para validação DFT: filtra estáveis e ordena por confiança
# "confiança" = margem acima do limiar (gap_hse_pred - 3.4)
df_new["margin"] = df_new["gap_hse_pred"] - UWBG_THRESHOLD

df_priority = df_new[
    (df_new["dyn_stab"] == "Yes") &   # estável dinamicamente
    (df_new["ehull"] < 0.1)           # próximo da convex hull
].sort_values("gap_hse_pred", ascending=False).reset_index(drop=True)

print(f"Candidatos novos com dyn_stab=Yes e ehull<0.1: {len(df_priority)}")
print()

# Distribuição por fórmula/estequiometria
stoich = df_priority["formula"].apply(
    lambda f: "binário" if len(set(c for c in f if c.isupper())) == 2
    else ("ternário" if len(set(c for c in f if c.isupper())) == 3 else "outro")
)
print("Por complexidade composicional:")
print(stoich.value_counts().to_string())
print()

# Scatter: PBE vs HSE predito para candidatos novos priorizados
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    df_priority["gap_pbe"],
    df_priority["gap_hse_pred"],
    c=df_priority["ehull"],
    cmap="viridis_r",
    s=40, alpha=0.8, edgecolors="white", lw=0.3
)
ax.axhline(UWBG_THRESHOLD, color="red", ls="--", lw=1.5, label="UWBG 3.4 eV")
ax.axvline(UWBG_THRESHOLD, color="red", ls=":", lw=1, alpha=0.5)
ax.plot([0, 12], [0, 12], "k--", lw=0.8, alpha=0.4, label="PBE = HSE (ideal)")
plt.colorbar(sc, ax=ax, label="ehull (eV/atom)")
ax.set_xlabel("Bandgap PBE (eV)")
ax.set_ylabel("Bandgap HSE predito (eV)")
ax.set_title("Candidatos novos priorizados para validação DFT\n(dyn_stab=Yes, ehull<0.1)")
ax.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / "figures/uwbg_new_candidates_priority.png", dpi=150)
plt.show()

# Exporta lista de prioridade para validação
export_cols = ["uid", "formula", "n_atoms", "gap_pbe", "gap_hse_pred", "margin", "ehull", "dyn_stab", "hform"]
df_priority[export_cols].to_csv(RUN_DIR / "outputs/uwbg_new_candidates_for_dft.csv", index=False)
print(f"Lista exportada: uwbg_new_candidates_for_dft.csv ({len(df_priority)} candidatos)")

## 9. Visualização dos top candidatos UWBG

In [ ]:
import py3Dmol
from pymatgen.io.cif import CifWriter
from IPython.display import display

top_viz = df[df["uwbg"]].head(5)

for row in top_viz.itertuples():
    struct   = structures[row.uid]
    hse_real = f"{row.gap_hse_true:.3f} eV" if row.gap_hse_true is not None else "N/A"

    print("=" * 60)
    print(f"UID       : {row.uid}")
    print(f"Fórmula   : {row.formula}  ({row.n_atoms} átomos)")
    print(f"HSE pred  : {row.gap_hse_pred:.3f} eV")
    print(f"HSE real  : {hse_real}")
    print(f"PBE real  : {row.gap_pbe:.3f} eV" if row.gap_pbe else "PBE real  : N/A")

    try:
        cif_str = CifWriter(struct).__str__()
        view = py3Dmol.view(width=550, height=350)
        view.addModel(cif_str, "cif")
        view.setStyle({"sphere": {"scale": 0.35}, "stick": {"radius": 0.12}})
        view.addUnitCell()
        view.zoomTo()
        display(view.show())
    except Exception as e:
        print(f"  [!] Visualização falhou: {e}")
    print()

## 10. Análise de Óxidos 2D

Óxidos são uma classe quimicamente distinta: a correção PBE→HSE é sistematicamente maior (~+1.5 a +3.0 eV) do que a média do dataset (~+0.7 eV), o que aumenta o risco de subestimação pelo modelo.

Esta seção analisa:
- Todos os óxidos no dataset de triagem (fórmula contém O)
- Subgrupo XO₂ (estequiometria simples: um metal + dois oxigênios)
- Viés sistemático: distribuição de Δ = HSE − PBE para óxidos vs restante
- Candidatos UWBG óxidos identificados pelo modelo

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

# --- Carrega dados (compatível com execução isolada) ---
try:
    _ = df
except NameError:
    df = pd.read_csv(RUN_DIR / "outputs/uwbg_candidates.csv")

# --- Classificação de óxidos ---
def has_oxygen(formula: str) -> bool:
    """Detecta óxidos: 'O' não seguido de 's' (Os=Osmium) ou 'g' (Og=Oganesson)."""
    return bool(re.search(r'O(?![sg])', formula))

def is_xo2(formula: str) -> bool:
    """XO₂: exatamente um elemento + dois oxigênios (ex: TiO2, GeO2, SnO2)."""
    return bool(re.fullmatch(r'[A-Z][a-z]?O2', formula))

df = df.copy()
df['is_oxide'] = df['formula'].apply(has_oxygen)
df['is_xo2']   = df['formula'].apply(is_xo2)

df_ox  = df[df['is_oxide']]
df_xo2 = df[df['is_xo2']]

print(f"Total no dataset de triagem : {len(df):>5}")
print(f"Óxidos (contém O)           : {len(df_ox):>5}  ({len(df_ox)/len(df)*100:.1f}%)")
print(f"  → UWBG predito            : {df_ox['uwbg'].sum():>5}")
print(f"XO₂ (estequiometria simples): {len(df_xo2):>5}")
print(f"  → UWBG predito            : {df_xo2['uwbg'].sum():>5}")

# ----------------------------------------------------------------
# Tabela: candidatos UWBG óxidos
# ----------------------------------------------------------------
df_ox_uwbg = df_ox[df_ox['uwbg']].sort_values('gap_hse_pred', ascending=False).copy()
df_ox_uwbg['subgrupo'] = df_ox_uwbg['is_xo2'].map({True: 'XO₂', False: 'outro óxido'})

display_cols = ['formula', 'subgrupo', 'gap_pbe', 'gap_hse_true', 'gap_hse_pred']
rename = {
    'formula': 'Fórmula', 'subgrupo': 'Subgrupo',
    'gap_pbe': 'PBE (eV)', 'gap_hse_true': 'HSE real (eV)', 'gap_hse_pred': 'HSE pred (eV)'
}
df_show = df_ox_uwbg[display_cols].rename(columns=rename)

styled = (
    df_show.style
    .format({'PBE (eV)': '{:.3f}', 'HSE real (eV)': '{:.3f}', 'HSE pred (eV)': '{:.3f}'}, na_rep='—')
    .background_gradient(subset=['HSE pred (eV)'], cmap='YlOrRd')
    .set_caption(f"Candidatos UWBG óxidos — {len(df_ox_uwbg)} materiais (HSE pred > 3.4 eV)")
)
display(styled)

# ----------------------------------------------------------------
# Fig 1: Scatter PBE vs HSE_pred — óxidos destacados
# ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Painel esquerdo: todos os materiais
ax = axes[0]
non_ox = df[~df['is_oxide']]
ax.scatter(non_ox['gap_pbe'], non_ox['gap_hse_pred'],
           s=8, alpha=0.3, color='steelblue', label='Não-óxidos', rasterized=True)
ax.scatter(df_ox['gap_pbe'], df_ox['gap_hse_pred'],
           s=20, alpha=0.7, color='tomato', label='Óxidos', rasterized=True)
ax.scatter(df_xo2['gap_pbe'], df_xo2['gap_hse_pred'],
           s=40, alpha=0.9, color='darkred', marker='*', label='XO₂', zorder=5)
ax.axhline(3.4, color='gray', ls='--', lw=1, label='UWBG 3.4 eV')
ax.set_xlabel('Gap PBE (eV)')
ax.set_ylabel('Gap HSE pred (eV)')
ax.set_title('PBE vs HSE predito — óxidos em destaque')
ax.legend(fontsize=8)

# Painel direito: distribuição de Δ = HSE_real - PBE (onde HSE real existe)
ax2 = axes[1]
df_with_hse = df.dropna(subset=['gap_hse_true'])
df_with_hse = df_with_hse.copy()
df_with_hse['delta'] = df_with_hse['gap_hse_true'] - df_with_hse['gap_pbe']

ox_delta   = df_with_hse[df_with_hse['is_oxide']]['delta']
nonox_delta = df_with_hse[~df_with_hse['is_oxide']]['delta']

bins = np.linspace(-1, 5, 50)
ax2.hist(nonox_delta, bins=bins, alpha=0.6, color='steelblue',
         label=f'Não-óxidos (n={len(nonox_delta)}, μ={nonox_delta.mean():.2f} eV)')
ax2.hist(ox_delta, bins=bins, alpha=0.7, color='tomato',
         label=f'Óxidos (n={len(ox_delta)}, μ={ox_delta.mean():.2f} eV)')
ax2.axvline(nonox_delta.mean(), color='steelblue', lw=2, ls='--')
ax2.axvline(ox_delta.mean(), color='darkred', lw=2, ls='--')
ax2.set_xlabel('Δ = HSE − PBE (eV)')
ax2.set_ylabel('Contagem')
ax2.set_title('Distribuição da correção HSE − PBE')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(RUN_DIR / "figures/oxide_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

# ----------------------------------------------------------------
# Estatísticas de Δ por subgrupo
# ----------------------------------------------------------------
print("\n=== Correção PBE → HSE por subgrupo (onde HSE real existe) ===")
for label, mask in [
    ('Não-óxidos', ~df_with_hse['is_oxide']),
    ('Óxidos',      df_with_hse['is_oxide']),
    ('XO₂',         df_with_hse['is_xo2']),
]:
    sub = df_with_hse[mask]['delta']
    if len(sub) == 0:
        continue
    print(f"  {label:<12}: n={len(sub):>4}  μ={sub.mean():.3f} eV  "
          f"σ={sub.std():.3f} eV  min={sub.min():.2f}  max={sub.max():.2f}")

## 11. Faixa de Perigo: Materiais com HSE entre 2–4 eV

Materiais com gap HSE entre 2 e 4 eV são os mais difíceis de predizer:
- O modelo interpola entre semicondutores comuns (bem representados) e a região UWBG (esparsa no treino)
- O erro de correção PBE→HSE é mais variável nessa região
- Falsos positivos e falsos negativos de classificação UWBG concentram-se aqui

Esta seção avalia o desempenho do modelo especificamente nessa faixa.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

df_val = df.dropna(subset=['gap_hse_true']).copy()
df_val['error']   = df_val['gap_hse_pred'] - df_val['gap_hse_true']   # com sinal
df_val['abs_err'] = df_val['error'].abs()

# ── Faixas granulares (1 eV cada) ────────────────────────────────
fine_edges  = [0, 1, 2, 3, 4, 5, 6, 7, 20]
fine_labels = ['0–1', '1–2', '2–3', '3–4', '4–5', '5–6', '6–7', '7+']
df_val['bin'] = pd.cut(df_val['gap_hse_true'], bins=fine_edges, labels=fine_labels, right=False)

# ── Tabela detalhada ──────────────────────────────────────────────
print("=== Erro de predição por faixa de 1 eV (gap HSE real) ===")
print(f"{'Faixa':>6} {'N':>5} {'MAE':>8} {'RMSE':>8} {'Viés':>8} {'Max↑':>8} {'Max↓':>8}")
print(f"{'(eV)':>6} {'':>5} {'(eV)':>8} {'(eV)':>8} {'(eV)':>8} {'(eV)':>8} {'(eV)':>8}")
print("-" * 60)
for lbl in fine_labels:
    sub = df_val[df_val['bin'] == lbl]
    if len(sub) < 5:
        continue
    mae  = sub['abs_err'].mean()
    rmse = np.sqrt((sub['error'] ** 2).mean())
    bias = sub['error'].mean()                      # negativo = subestimação
    max_over  = sub['error'].max()                  # maior superestimação
    max_under = sub['error'].min()                  # maior subestimação
    print(f"  {lbl:>4}  {len(sub):>5} {mae:>8.3f} {rmse:>8.3f} {bias:>+8.3f} {max_over:>+8.3f} {max_under:>+8.3f}")
print("-" * 60)
bias_total = df_val['error'].mean()
mae_total  = df_val['abs_err'].mean()
rmse_total = np.sqrt((df_val['error'] ** 2).mean())
print(f"  {'Total':>4}  {len(df_val):>5} {mae_total:>8.3f} {rmse_total:>8.3f} {bias_total:>+8.3f}")

# ── Figura: 3 painéis ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Scatter pred vs real (3 faixas grandes por cor)
ax = axes[0]
coarse_edges  = [0, 2, 4, 20]
coarse_labels = ['0–2 eV (semicond.)', '2–4 eV (perigo)', '4+ eV (UWBG)']
coarse_colors = {'0–2 eV (semicond.)': 'steelblue',
                 '2–4 eV (perigo)':    'darkorange',
                 '4+ eV (UWBG)':       'seagreen'}
df_val['range'] = pd.cut(df_val['gap_hse_true'], bins=coarse_edges,
                         labels=coarse_labels, right=False)
for lbl, color in coarse_colors.items():
    sub = df_val[df_val['range'] == lbl]
    ax.scatter(sub['gap_hse_true'], sub['gap_hse_pred'],
               s=8, alpha=0.4, color=color, label=lbl, rasterized=True)
lims = [df_val['gap_hse_true'].min() - 0.3, df_val['gap_hse_true'].max() + 0.3]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel('HSE real (eV)'); ax.set_ylabel('HSE predito (eV)')
ax.set_title('Predito vs Real')
ax.legend(fontsize=8)

# 2. Viés médio por faixa de 1 eV (regressão à média)
ax2 = axes[1]
bin_stats = (df_val.groupby('bin', observed=True)['error']
             .agg(bias='mean', std='std', n='count')
             .reset_index())
bin_stats = bin_stats[bin_stats['n'] >= 5]
x = range(len(bin_stats))
ax2.bar(x, bin_stats['bias'], color=['tomato' if b < 0 else 'steelblue'
                                      for b in bin_stats['bias']], alpha=0.7)
ax2.errorbar(x, bin_stats['bias'], yerr=bin_stats['std'] / np.sqrt(bin_stats['n']),
             fmt='none', color='black', capsize=4)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xticks(list(x))
ax2.set_xticklabels(bin_stats['bin'].tolist(), rotation=45, ha='right')
ax2.set_ylabel('Viés médio (eV)')
ax2.set_title('Viés por faixa — negativo = subestimação')
for xi, (_, row) in zip(x, bin_stats.iterrows()):
    ax2.text(xi, row['bias'] + (0.02 if row['bias'] >= 0 else -0.06),
             f"n={int(row['n'])}", ha='center', fontsize=7)

# 3. Distribuição do erro absoluto por faixa (violin)
ax3 = axes[2]
bins_with_data = [lbl for lbl in fine_labels
                  if len(df_val[df_val['bin'] == lbl]) >= 5]
data_v = [df_val[df_val['bin'] == lbl]['abs_err'].values for lbl in bins_with_data]
vp = ax3.violinplot(data_v, positions=range(len(bins_with_data)),
                    showmedians=True, showextrema=False)
for body in vp['bodies']:
    body.set_alpha(0.6)
ax3.set_xticks(range(len(bins_with_data)))
ax3.set_xticklabels(bins_with_data, rotation=45, ha='right')
ax3.set_ylabel('Erro absoluto (eV)')
ax3.set_title('Distribuição do erro por faixa')

plt.tight_layout()
plt.savefig(RUN_DIR / "figures/danger_zone_analysis.png", dpi=150, bbox_inches='tight')
plt.show()

# ── Top erros na faixa de perigo ──────────────────────────────────
df_danger = (df_val[df_val['range'] == '2–4 eV (perigo)']
             .sort_values('abs_err', ascending=False))
print(f"\n=== Top 15 maiores erros na faixa 2–4 eV ===")
print(f"{'Fórmula':15} {'HSE real':>10} {'HSE pred':>10} {'Erro (eV)':>10} {'Óxido?':>8}")
print("-" * 58)
for _, row in df_danger.head(15).iterrows():
    print(f"  {row['formula']:13} {row['gap_hse_true']:>10.3f} {row['gap_hse_pred']:>10.3f} "
          f"{row['error']:>+10.3f} {'sim' if row['is_oxide'] else 'não':>8}")


## 12. Conclusão

Pipeline implementado end-to-end:

1. **C2DB** — 16905 materiais 2D, 12062 amostras multi-fidelidade (8699 PBE + 3363 HSE)
2. **MEGNet multi-fidelidade** — `gap` (PBE=0) e `gap_hse` (HSE=1) como fidelidades
3. **Transfer learning** — 106/106 pesos compatíveis transferidos do MEGNet MP
4. **Triagem UWBG** — predição HSE para todos os candidatos, filtro Eg > 3,4 eV
5. **Avaliação como classificador** — precision, recall, F1 e curva P-R no regime UWBG
6. **Candidatos novos** — materiais com UWBG predito e sem HSE calculado, priorizados por estabilidade e exportados para validação DFT (`uwbg_new_candidates_for_dft.csv`)

In [ ]:
import shutil

nb_src = NOTEBOOK_DIR / "megnet_finetune_c2db.ipynb"
nb_dst = RUN_DIR / "notebook.ipynb"
if nb_src.exists():
    shutil.copy2(nb_src, nb_dst)
    print(f"Notebook salvo em: {nb_dst.resolve()}")
else:
    print(f"Notebook fonte não encontrado: {nb_src}")